# Experimento 01 — DummyClassifier

Línea base mayoritaria; no requiere comparar V1–V4.

Sólo se utilizan train y validation. Test no se carga.

In [1]:
from pathlib import Path
import json,time
from datetime import datetime,timezone
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score,balanced_accuracy_score,classification_report,confusion_matrix,f1_score,precision_score,recall_score
ROOT=Path.cwd().parent if Path.cwd().name=="experimentos" else Path.cwd()
DATA=ROOT/"data/processed/modelado/dataset_modelado_nivel_actividad_firms.parquet"
SPLITS=ROOT/"data/processed/modelado/splits";OUT=ROOT/"results/experimentos_parciales";OUT.mkdir(parents=True,exist_ok=True)
datos=pd.read_parquet(DATA);datos.fecha_inicio_semana=pd.to_datetime(datos.fecha_inicio_semana)
KEY=["departamento","fecha_inicio_semana"];TARGET="nivel_actividad_firms";ORDER=["Sin actividad","Bajo","Moderado","Alto"];RS=42
def subset(name):
 k=pd.read_parquet(SPLITS/f"{name}_keys.parquet");k.fecha_inicio_semana=pd.to_datetime(k.fecha_inicio_semana)
 return datos.merge(k,on=KEY,how="inner",validate="one_to_one")
train=subset("train");validation=subset("validation")
assert len(train)==5546 and len(validation)==1188
CAT=["departamento","mes"]
MET=["temperature_2m_max_mean_weekly","relative_humidity_2m_min_mean_weekly","wind_speed_10m_max_mean_weekly","precipitation_sum_weekly"]
LAGS=[f"cantidad_detecciones_lag{i}" for i in range(1,5)]
ROLL=["promedio_detecciones_ultimas_4_semanas","max_detecciones_ultimas_4_semanas","semanas_con_actividad_ultimas_4"]
VARIANTS={"V1":CAT+MET,"V2":CAT+MET+LAGS,"V3":CAT+MET+ROLL,"V4":CAT+MET+LAGS+ROLL}
PROHIBITED={"cantidad_detecciones","fecha_inicio_semana","anio","numero_semana",TARGET,"nivel_actividad_firms_codigo"}
assert all(PROHIBITED.isdisjoint(v) for v in VARIANTS.values())
def metrics(y,p):
 r=classification_report(y,p,labels=ORDER,output_dict=True,zero_division=0)
 return {"accuracy_validation":accuracy_score(y,p),"balanced_accuracy_validation":balanced_accuracy_score(y,p),"precision_macro_validation":precision_score(y,p,average="macro",zero_division=0),"recall_macro_validation":recall_score(y,p,average="macro",zero_division=0),"f1_macro_validation":f1_score(y,p,average="macro",zero_division=0),"f1_weighted_validation":f1_score(y,p,average="weighted",zero_division=0),**{f"precision_{x.lower().replace(' ','_')}":r[x]["precision"] for x in ORDER},**{f"recall_{x.lower().replace(' ','_')}":r[x]["recall"] for x in ORDER},**{f"f1_{x.lower().replace(' ','_')}":r[x]["f1-score"] for x in ORDER}}
def row(eid,model,variant,params,pred,seconds,note):return {"experiment_id":eid,"modelo":model,"variante_variables":variant,"hiperparametros":json.dumps(params,sort_keys=True,ensure_ascii=False),**metrics(validation[TARGET],pred),"duracion_segundos":seconds,"fecha_ejecucion":datetime.now(timezone.utc).isoformat(),"notebook_origen":NOTEBOOK,"observaciones":note}

from sklearn.dummy import DummyClassifier
NOTEBOOK="Experimento_01_Dummy.ipynb"

In [2]:
print(train[TARGET].value_counts());print(validation[TARGET].value_counts())
t0=time.perf_counter();model=DummyClassifier(strategy="most_frequent",random_state=RS).fit(np.ones((len(train),1)),train[TARGET]);pred=model.predict(np.ones((len(validation),1)));elapsed=time.perf_counter()-t0
results=pd.DataFrame([row("dummy_most_frequent","DummyClassifier","N/A",{"strategy":"most_frequent","random_state":RS},pred,elapsed,"Baseline; no usa predictores")]);results.round(4)

nivel_actividad_firms
Sin actividad    3477
Moderado          886
Bajo              866
Alto              317
Name: count, dtype: int64
nivel_actividad_firms
Sin actividad    745
Moderado         190
Bajo             185
Alto              68
Name: count, dtype: int64


,experiment_id,modelo,variante_variables,hiperparametros,accuracy_validation,balanced_accuracy_validation,precision_macro_validation,recall_macro_validation,f1_macro_validation,f1_weighted_validation,...,recall_moderado,recall_alto,f1_sin_actividad,f1_bajo,f1_moderado,f1_alto,duracion_segundos,fecha_ejecucion,notebook_origen,observaciones
0,dummy_most_frequent,DummyClassifier,N/A,"{""random_state"": 42, ""strategy"": ""most_frequent""}",0.6271,0.25,0.1568,0.25,0.1927,0.4834,...,0.0,0.0,0.7708,0.0,0.0,0.0,0.0051,2026-09-30T19:03:45.478322+00:00,Experimento_01_Dummy.ipynb,Baseline; no usa predictores


In [3]:
cm=confusion_matrix(validation[TARGET],pred,labels=ORDER);display(pd.DataFrame(cm,index=ORDER,columns=ORDER));display(pd.DataFrame(classification_report(validation[TARGET],pred,labels=ORDER,output_dict=True,zero_division=0)).T.loc[ORDER].round(4))

,Sin actividad,Bajo,Moderado,Alto
Sin actividad,745,0,0,0
Bajo,185,0,0,0
Moderado,190,0,0,0
Alto,68,0,0,0


,precision,recall,f1-score,support
Sin actividad,0.6271,1.0,0.7708,745.0
Bajo,0.0000,0.0,0.0000,185.0
Moderado,0.0000,0.0,0.0000,190.0
Alto,0.0000,0.0,0.0000,68.0


In [4]:
results.to_csv(OUT/"01_dummy.csv",index=False);print("Guardado",OUT/"01_dummy.csv")

Guardado /home/pepo/Materias/PAA/results/experimentos_parciales/01_dummy.csv
